In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
import os
import polars as pl
import polars.selectors as cs
import seaborn as sns
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from ucimlrepo import fetch_ucirepo

sys.path.insert(0, str(Path().resolve().parents[1]))
from config import cfg

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

heart_disease = fetch_ucirepo(id=45)
raw = pl.from_pandas(heart_disease.data.original)

In [ ]:
OUTLIER_THRESHOLD = 2  # Standard deviations from the mean

data = raw.drop_nulls()
data = data.with_columns(
    (pl.col("num") >= 1).cast(pl.Int64).alias("num"),
    cs.exclude("sex", "fbs", "exang", "num").map_batches(lambda col: (col - col.mean()) / col.std()),
)
data = data.filter(pl.all_horizontal(cs.exclude("sex", "fbs", "exang", "num").abs() <= OUTLIER_THRESHOLD))
data.describe()

In [ ]:
fig, ax = plt.subplots(1, figsize=(17, 4))
ax = sns.boxplot(data=data)
plt.show()

In [ ]:
batch_size = 8
X = data[:, :-1].to_torch().float()
y = data[:, -1].to_torch().float()[:, None]  # make it a column vector
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=test_dataset.tensors[0].shape[0], shuffle=False)
print(X.shape, y.shape)

# Neural Network

In [ ]:
class HeartDiseaseNN(nn.Module):
    def __init__(self, input_size, dropout_rate=0.2):
        super().__init__()
        self.dr = dropout_rate
        self.input = nn.Linear(input_size, 32)
        self.fc1 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = F.relu(self.input(x))
        x = F.dropout(x, p=self.dr, training=self.training)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dr, training=self.training)
        return self.output(x)

In [ ]:
# test the model on a bit of data
net = HeartDiseaseNN(13)

X, y = next(iter(train_loader))
y_hat = net(X)
print(y_hat.shape, y.shape)

# test the loss function
lossfun = nn.BCEWithLogitsLoss()
lossfun(y_hat, y)

In [ ]:
net = HeartDiseaseNN(13, dropout_rate=0.2)
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001, weight_decay=0.01)
lossfun = nn.BCEWithLogitsLoss()
num_epochs = 500

train_loss = torch.zeros(num_epochs)
test_loss = torch.zeros(num_epochs)
train_acc = torch.zeros(num_epochs)
test_acc = torch.zeros(num_epochs)


best_test_acc = 0.0
best_model_path = Path(f'{cfg.DATA_LAKE}/public/models/medical/heart_disease/heart_disease_UCI_model.pt')
best_model_path.parent.mkdir(parents=True, exist_ok=True)

for epoch_i in tqdm(range(num_epochs), desc="Training"):
    batch_loss = []

    for X, y in train_loader:
        y_hat = net(X)
        loss = lossfun(y_hat, y)

        # Optimize memory bandwidth and footprint
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_loss.append(loss.item())

        y_pred = torch.sigmoid(y_hat) >= 0.5
        train_acc[epoch_i] = 100 * (y_pred == y).float().mean().item()

    train_loss[epoch_i] = torch.tensor(batch_loss).mean()

    with torch.no_grad():
        X, y = next(iter(test_loader))
        y_hat = net(X)
        loss = lossfun(y_hat, y)
        test_loss[epoch_i] = loss.item()

        predictions = (torch.sigmoid(y_hat) >= 0.5).float()
        test_acc[epoch_i] = 100 * (predictions == y).float().mean().item()

        # Save best model
        if test_acc[epoch_i] > best_test_acc:
            best_test_acc = test_acc[epoch_i].item()
            torch.save(net.state_dict(), best_model_path)
            print(f"Saved new best model at epoch {epoch_i+1} with test acc: {best_test_acc:.2f}%")
best_model_path_with_acc = Path(f'{cfg.DATA_LAKE}/public/models/medical/heart_disease/heart_disease_UCI_model_{best_test_acc:.0f}percent.pt')
os.rename(best_model_path, best_model_path_with_acc)
print(f"Highest test accuracy: {best_test_acc:.2f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

ax[0].plot(train_loss, "s-", label="Train")
ax[0].plot(test_loss, "s-", label="Test")
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[0].set_title("Model loss")

ax[1].plot(train_acc, "s-", label="Train")
ax[1].plot(test_acc, "o-", label="Test")
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Accuracy (%)")
ax[1].set_title(f"Final model test accuracy: {test_acc[-1]:.2f}%")
ax[1].legend()

plt.show()